In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
import ollama
import re

/tmp/ipykernel_13913/3676925187.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


fetch data

In [2]:
def cleanText(text):
    text = re.sub(r'-\n(\w)', r'\1', text)     # re-join words hyphenated across lines
    text = text.replace('•', '\n- ')           # bullets as their own lines
    text = re.sub(r'[ \t]+', ' ', text)        # collapse spaces/tabs only
    text = re.sub(r'\n{3,}', '\n\n', text)     # cap blank lines
    return text.strip()

In [3]:
loader = PyPDFLoader("data/kejriwal-et-al-2019-knowledge-graphs-construction-management-and-querying.pdf")
pages = loader.load()

In [4]:
for page in pages: 
    page.page_content = cleanText(page.page_content) 

split into chunks

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 800, chunk_overlap= 100) # created an instance of the text splitter
chunks = splitter.split_documents(pages)

vectorize 

In [6]:
# functions & global variables 

def createEmbedding(text):
    response = ollama.embed(
    model='nomic-embed-text-v2-moe',
    input=text,
    )
    return response.embeddings[0]

In [7]:
doc_embeddings = [] # this is our list of embeddings from out document

for chunk in chunks:
    chunk_embedding = createEmbedding(chunk.page_content)
    doc_embeddings.append(chunk_embedding) 

pass data into llm to disover relationships and nodes,  The entity and relationship types would become labels  and relationship types in the graph

In [8]:
from langchain_ollama import ChatOllama
from langchain_neo4j import Neo4jGraph, LLMGraphTransformer


In [27]:
llm = ChatOllama(model="deepseek-r1:1.5b ", temperature=0)

In [18]:
graph = Neo4jGraph(url="neo4j://127.0.0.1:7687", username="neo4j", password="neo4j123")

In [28]:
transformer = LLMGraphTransformer(llm=llm)

In [22]:
chunks[0].page_content

'Semantic Web 10 (2019) 961–962 961\nDOI 10.3233/SW-190370\nIOS Press\nKnowledge graphs: Construction,\nmanagement and querying\nEditorial\nMayank Kejriwal a,*, Juan Sequeda b and Vanessa Lopezc\na Information Sciences Institute, University of Southern California, Los Angeles, CA, USA\nE-mail: kejriwal@isi.edu\nb data.world, Austin, TX, USA\nE-mail: juan@data.world\nc IBM, Dublin, Ireland\nE-mail: vanlopez@ie.ibm.com\nKeywords: Knowledge graphs, knowledge graph querying, knowledge graph construction, applications, linked data, knowledge\ngraph management\nGraphs have been one of the fundamental data models in Artiﬁcial Intelligence and Databases, but the\nastounding growth of the Web has brought them to'

In [32]:
graph_documents = transformer.convert_to_graph_documents([chunks[0]])

In [36]:
type(graph_documents[0]) 

langchain_neo4j.graphs.graph_document.GraphDocument

In [43]:
graph_documents[0].relationships

[Relationship(source=Node(id='Knowledge Graph Construction', type='Knowledge graph', properties={}), target=Node(id='Knowledge Graph Management', type='Knowledge graph', properties={}), type='CONSTRUCTION,_MANAGEMENT_AND_QUERYING', properties={}),
 Relationship(source=Node(id='Knowledge Graph Applications', type='Knowledge graph', properties={}), target=Node(id='Applications', type='Knowledge graph', properties={}), type='DATA_INTEGRATION,_DECISION-MAKING,_KNOWLEDGE_SHARING', properties={}),
 Relationship(source=Node(id='Linked Data', type='Linked data', properties={}), target=Node(id='Knowledge Graph Management', type='Knowledge graph', properties={}), type='ENHANCING_MANAGEMENT', properties={}),
 Relationship(source=Node(id='Large-Scale Knowledge Graphs', type='Knowledge graph', properties={}), target=Node(id='Knowledge Graphs', type='Management', properties={}), type='RELEVANCE_IN_DIGITAL_AGE', properties={})]

In [50]:
from neo4j_viz import Node, Relationship, VisualizationGraph

VG = VisualizationGraph(nodes=graph_documents[0].nodes, relationships=graph_documents[0].relationships)

VG.render(initial_zoom=2)

AttributeError: 'Relationship' object has no attribute 'id'